# Cryptic crossword pipeline`clue` → **T5** generates k (answer, reason) candidates → **DeBERTa** scores eachone → return the highest-scoring pair.This notebook is a **thin wrapper**. All the logic lives in `code/pipeline/*.py`,so a fix never has to be applied in two places, and the same code runs here, fromthe CLI, and inside a Slurm job. If you want to change behaviour, edit the moduleor the config — not a copy of the code pasted into a cell.| Want to | Do ||---|---|| solve one clue | section 2 || see what the generator actually emits | section 3 || evaluate on N clues with metrics + plots | section 4 || swap either model | section 5 || run 10k clues | don't — use Slurm, section 6 |

## 1. SetupAdds `code/` to `sys.path` so `import pipeline` works. The package name is`pipeline`, **not** `code.pipeline` — `code` is a Python standard-library modulename and shadowing it breaks unrelated imports.

In [ ]:
import sysfrom pathlib import PathREPO = Path.cwd()while not (REPO / "united-cryptonite-wordplay-dataset").is_dir() and REPO != REPO.parent:    REPO = REPO.parent          # works whether the kernel started at the repo root or in code/sys.path.insert(0, str(REPO / "code"))import pipelinefrom pipeline import config_io, data, metrics as metrics_mod, pathsfrom pipeline import pipeline as pipeprint("repo:      ", REPO)print("storage:   ", paths.storage_root())print("on cluster:", paths.on_cluster())

In [ ]:
# The resolved config, after `extends` and any override. Everything the two# stages do is in here -- read it once and the pipeline holds no surprises.cfg = config_io.load_config("t5large_deberta_large")print("generator:", cfg["generator"]["model"])print("           ", cfg["generator"]["strategy"], "k =", cfg["generator"]["num_candidates"])print("scorer:   ", cfg["scorer"]["model"])print("selection:", cfg["selection"])

> **The scorer checkpoint is pending.** Until the fine-tuned DeBERTa is dropped> in (see `models/README.md`), `scorer.model` resolves to the *untrained* base> `microsoft/deberta-v3-small`, whose classification head is random — **its scores> are noise**. The pipeline raises a `RuntimeWarning` and stamps> `scorer_head_untrained: true` into `metrics.json` when that happens. Any> accuracy from such a run is void; the plumbing still works, which is the point.>> Swapping it in later changes **one config line**, no code.

## 2. Solve one clue`pipeline.solve` prints the full stage-by-stage trace and caches the loaded modelsacross calls, so only the first call pays the load cost.

In [ ]:
record, best = pipeline.solve("attack general at end of month (6)", config=cfg)print()print("ANSWER:", best.answer if best else "(none)")print("REASON:", best.reason if best else "(none)")

In [ ]:
# A clue whose answer you know, so the trace can mark correctness.record, best = pipeline.solve("make progress socially in stated region (5)", config=cfg)

## 3. What does the generator actually emit?Run this **first** against any unfamiliar checkpoint. The parser turns a rawdecode into `(answer, reason)` using `generator.parse.patterns`, and a wrongpattern does not raise — it silently yields empty reasons, leaving the scorerjudging `(clue, answer)` only. This cell shows the truth.

In [ ]:
from pipeline.adapters import build_generatorgen = build_generator(cfg["generator"])probe_records = data.load_split("test", limit=3, seed=0)for rec, decodes in gen.probe(probe_records):    print("=" * 90)    print("CLUE  ", rec.clue)    print("GOLD  ", rec.gold_answer)    for i, text in enumerate(decodes):        answer, reason = gen.parse_output(text, rec.enumeration)        print(f"  [{i:2d}] raw={text!r}")        print(f"       -> answer={answer!r}  reason={reason!r}")print()print(gen.stats, "<- reason_parsed == 0 means the patterns need fixing (or the model emits answers only)")

## 4. Evaluate on a batchRead the four accuracy numbers together:- **baseline** — the generator's own top-1. What "a simple LLM generator" scores.- **pipeline** — the reranked pick. The thing being claimed as better.- **oracle@k** — was the gold answer *anywhere* in the k candidates? A hard  ceiling: a scorer cannot pick an answer that was never proposed. If oracle@k is  barely above baseline, the **generator** is the bottleneck and no amount of  scorer work will help.- **random@k** — the floor. A "reranker" landing here has learned nothing.Plus a McNemar p-value, because at n=100 a few points of difference is noise.

In [ ]:
N = 100          # keep it small in a notebook; use Slurm past ~1000records = data.load_split("test", limit=N, seed=42)run_dir = paths.runs_root() / "notebook-demo"outcome = pipe.run(records, cfg, run_dir, print_first=2, progress_every=25)result = metrics_mod.compute_metrics(outcome.records, cfg["selection"])result["filters"] = outcome.filter_stats.as_dict()result["scorer_head_untrained"] = outcome.scorer_head_untrainedprint(metrics_mod.format_metrics(result))if outcome.scorer_head_untrained:    print("!! scorer head was untrained -- the numbers above are noise, not a result")

In [ ]:
# Figures land in the run directory; display them inline too.from IPython.display import Image, displayfor path in metrics_mod.write_plots(outcome.records, result, run_dir / "figures"):    print(path.name)    display(Image(filename=str(path)))

In [ ]:
# Where did reranking win, and where did it break something that was right?import csvmetrics_mod.write_predictions_csv(outcome.records, run_dir / "predictions.csv", cfg["selection"])rows = list(csv.DictReader(open(run_dir / "predictions.csv")))wins   = [r for r in rows if r["pipeline_correct"] == "1" and r["baseline_correct"] == "0"]losses = [r for r in rows if r["pipeline_correct"] == "0" and r["baseline_correct"] == "1"]print(f"{len(wins)} wins, {len(losses)} losses out of {len(rows)} clues\n")for label, group in (("WIN", wins[:5]), ("LOSS", losses[:5])):    for r in group:        print(f"[{label}] {r['clue']}")        print(f"        gold={r['gold_answer']!r} baseline={r['baseline_answer']!r} "              f"pipeline={r['pipeline_answer']!r} score={r['pipeline_score']}")    print()

### Does the scorer prefer *human* reasoning to the model's?A direct sanity check on the scorer, and the generalization of the earlier1000-clue POC. Needs `wordplay` annotations, so **val, not test** — the test splithas none by construction.

In [ ]:
from pipeline.cli import evaluate_gold_reasonsannotated = data.load_split("val", limit=60, seed=7, require_reason=True)ann_outcome = pipe.run(annotated, cfg, paths.runs_root() / "notebook-gold", print_first=0)gold_eval = evaluate_gold_reasons(ann_outcome.records, cfg)for k, v in (gold_eval or {}).items():    print(f"{k:34s} {v}")

## 5. Swapping either modelNo code change. Either edit the `model` field in`code/pipeline/config/t5large_deberta_large.json`, or override it per run:

In [ ]:
# A bigger scorer (needs a smaller batch), against the same candidates.alt = config_io.apply_overrides(cfg, [    "scorer.model=microsoft/deberta-v3-large",    "scorer.batch_size=16",])print("scorer now:", alt["scorer"]["model"])# More, more-diverse candidates: raises oracle@k (the ceiling) at linear cost.alt = config_io.apply_overrides(cfg, [    "generator.num_candidates=20",    "generator.strategy=beam_sample_union",    "generator.beam_candidates=8",])print("generator now:", alt["generator"]["num_candidates"], "candidates")# The ablation that proves the scorer is doing the work: same code path, but# selection ignores the scorer entirely and reproduces the baseline.ablation = config_io.apply_overrides(cfg, ["selection.combine=generator"])print("selection now:", ablation["selection"]["combine"])

In [ ]:
# Re-score an EXISTING generation with a different scorer -- no regeneration.# Generation is by far the expensive half, so this is how you A/B scorers.rescored = pipe.run(    records,    alt,    paths.runs_root() / "notebook-rescore",    stage="score",    candidates_from=run_dir,    print_first=0,)print(metrics_mod.format_metrics(metrics_mod.compute_metrics(rescored.records, alt["selection"])))

## 6. Scaling up — use Slurm, not this notebookA notebook kernel on a preemptible node will lose a 10k run. The CLI streams eachfinished clue to `records.jsonl` and resumes from there, so a requeued jobcontinues instead of restarting.```bash# one shard at a timesbatch code/pipeline/run_pipeline.sbatch --split test --limit 1000# 10k across 4 parallel jobsfor i in 0 1 2 3; do  sbatch code/pipeline/run_pipeline.sbatch --split test --limit 10000 --shard $i/4done```Generation dominates the cost — roughly 1–3 clues/s for flan-t5-large with 12candidates (beam+sampling union) on one card. Scoring 12 candidates × 10k clues with DeBERTa-large isminutes by comparison. See `code/pipeline/README.md`.